In [ ]:
!pip install -q protobuf==5.29.6 kaggle-benchmarks numpy 2>/dev/null


## Wisconsin Card Sort Test (WCST) Analogue — Executive Functions

### Cognitive Science Rationale

The **Wisconsin Card Sorting Test** (Berg, 1948; Milner, 1963) is the gold standard for measuring **set-shifting** and **cognitive flexibility** — core executive functions. The model must:

1. **Infer** the active sorting rule from feedback patterns (rule is never stated)
2. **Detect** when the rule silently changes (feedback shifts from Correct to Incorrect)
3. **Adapt** by inferring and applying the new rule

**Perseveration** — continuing to apply the old rule after a shift — is the key metric. Frontal lobe patients show elevated perseveration (Milner, 1963). Miyake et al. (2000) established set-shifting as a distinct executive function.

### Design

- 6 blocks with 3 sorting dimensions (color, shape, number)
- Block 1: pure rule learning from examples
- Blocks 2-6: rule shifts with varying signal strength (easy→hard)
- History shows Correct/Incorrect feedback; model must infer new rule
- 35 test trials total, 30 post-shift

### Score
`0.25 × accuracy + 0.45 × (1 - perseveration_rate) + 0.30 × categories_completed_norm`

### References
- Berg, E.A. (1948). A simple objective technique for measuring flexibility in thinking.
- Milner, B. (1963). Effects of different brain lesions on card sorting.
- Miyake, A. et al. (2000). The unity and diversity of executive functions.


In [ ]:
import kaggle_benchmarks as kbench
import json as _json
import re
import random
import numpy as np

def _safe_log(data): print(_json.dumps(data, indent=2, default=str))

# ─── Stimuli Generation (inlined) ──────────────────────────────────

COLORS = ["red", "blue", "green", "yellow"]
SHAPES = ["circle", "triangle", "square", "star"]
NUMBERS = [1, 2, 3, 4]

REFERENCE_CARDS = [
    {"color": "red",    "shape": "circle",   "number": 1},
    {"color": "blue",   "shape": "triangle", "number": 2},
    {"color": "green",  "shape": "square",   "number": 3},
    {"color": "yellow", "shape": "star",     "number": 4},
]

RULES = ["color", "shape", "number"]
DIM_TO_IDX = {
    "color":  {v: i for i, v in enumerate(COLORS)},
    "shape":  {v: i for i, v in enumerate(SHAPES)},
    "number": {v: i for i, v in enumerate(NUMBERS)},
}


def card_str(card):
    n = card["number"]
    return f"{n} {card['color']} {card['shape']}{'s' if n > 1 else ''}"


def _correct_ref(target, rule):
    return DIM_TO_IDX[rule][target[rule]] + 1


def _make_target(active_rule, correct_ref_idx, rng):
    refs = REFERENCE_CARDS
    other_indices = [i for i in range(4) if i != correct_ref_idx]
    rng.shuffle(other_indices)
    target = {}
    dim_order = [active_rule] + [d for d in RULES if d != active_rule]
    for i, dim in enumerate(dim_order):
        if dim == active_rule:
            target[dim] = refs[correct_ref_idx][dim]
        else:
            pick_idx = other_indices[i - 1] if i - 1 < len(other_indices) else other_indices[0]
            target[dim] = refs[pick_idx][dim]
    return target


def _make_trial(rule, rng):
    correct_ref_idx = rng.randint(0, 3)
    target = _make_target(rule, correct_ref_idx, rng)
    return {"target": target, "active_rule": rule, "correct_answer": correct_ref_idx + 1}


def generate_wcst_blocks(seed=42):
    rng = random.Random(seed)
    rule_transitions = [
        ("color", "shape"), ("shape", "number"), ("number", "color"),
        ("color", "number"), ("number", "shape"),
    ]
    blocks = []
    # Block 1: Pure learning
    init_rule = "color"
    history = []
    for _ in range(6):
        t = _make_trial(init_rule, rng)
        history.append({**t, "response": t["correct_answer"], "feedback": "Correct"})
    test = [{**_make_trial(init_rule, rng), "is_post_shift": False, "prev_rule": None} for _ in range(5)]
    blocks.append({"block_id": 1, "history": history, "test_trials": test})
    # Blocks 2-6: Shift blocks
    for i, (old_rule, new_rule) in enumerate(rule_transitions):
        history = []
        for _ in range(5):
            t = _make_trial(old_rule, rng)
            history.append({**t, "response": t["correct_answer"], "feedback": "Correct"})
        n_incorrect = 2 if i < 2 else 3
        for _ in range(n_incorrect):
            t = _make_trial(new_rule, rng)
            old_ans = _correct_ref(t["target"], old_rule)
            history.append({**t, "response": old_ans, "feedback": "Incorrect"})
        n_hints = 3 if i < 2 else 2 if i < 4 else 1
        for _ in range(n_hints):
            t = _make_trial(new_rule, rng)
            history.append({**t, "response": t["correct_answer"], "feedback": "Correct"})
        test = [{**_make_trial(new_rule, rng), "is_post_shift": True, "prev_rule": old_rule} for _ in range(6)]
        blocks.append({"block_id": i + 2, "history": history, "test_trials": test})
    return {"reference_cards": REFERENCE_CARDS, "blocks": blocks,
            "total_test_trials": sum(len(b["test_trials"]) for b in blocks)}


WCST_BLOCKS = generate_wcst_blocks(seed=42)


# ─── Prompt Formatting ─────────────────────────────────────────────

def _format_block_prompt(block):
    refs = REFERENCE_CARDS
    lines = ["You are taking a card sorting test. There are 4 reference cards:"]
    for i, r in enumerate(refs, 1):
        lines.append(f"  Card {i}: {card_str(r)}")
    lines.append("")
    lines.append("For each trial, a target card must be matched to one of the 4 reference cards. "
                 "The matching rule is based on ONE dimension (color, shape, or number), but "
                 "the rule is NOT stated. You must figure it out from the feedback pattern below.")
    lines.append("")
    lines.append("IMPORTANT: The sorting rule may change. Pay close attention to when "
                 "responses start getting 'Incorrect' feedback — that means the rule "
                 "has changed and you need to figure out the NEW rule.")
    lines.append("")
    if block["history"]:
        lines.append("=== Previous trials (with responses and feedback) ===")
        for h in block["history"]:
            lines.append(f"Target: {card_str(h['target'])} → Response: Card {h['response']} → {h['feedback']}")
        lines.append("")
    n_test = len(block["test_trials"])
    lines.append(f"=== Your turn: sort the next {n_test} cards ===")
    lines.append(f"Based on the feedback pattern above, determine the CURRENT sorting rule "
                 f"and sort each card. Respond with EXACTLY {n_test} numbers (1-4), one per line.")
    lines.append("")
    for i, t in enumerate(block["test_trials"], 1):
        lines.append(f"Card {i}: {card_str(t['target'])}")
    lines.append("")
    lines.append(f"Your {n_test} answers (one number 1-4 per line):")
    return "\n".join(lines)


def _parse_responses(raw, n_expected):
    # Strategy 1: Lines that are just a single digit 1-4
    line_answers = []
    for line in raw.strip().split('\n'):
        line = line.strip()
        if re.match(r'^[1-4]$', line):
            line_answers.append(int(line))
    if len(line_answers) >= n_expected:
        return line_answers[-n_expected:]
    # Strategy 2: Last N standalone digits
    all_nums = re.findall(r'\b([1-4])\b', raw)
    if len(all_nums) >= n_expected:
        return [int(n) for n in all_nums[-n_expected:]]
    # Pad
    choices = [int(n) for n in all_nums]
    rng = np.random.RandomState(99)
    while len(choices) < n_expected:
        choices.append(int(rng.randint(1, 5)))
    return choices


# ─── The Benchmark Task ────────────────────────────────────────────

@kbench.task(name="Wisconsin Card Sorting (WCST)")
def exec_func_wcst(llm) -> float:
    """Wisconsin Card Sort Analogue (batch-prompt version).
    Tests cognitive flexibility through rule inference and set-shifting.
    Score = 0.25*accuracy + 0.45*(1-perseveration_rate) + 0.30*categories_norm
    Cognitive Science: Berg (1948), Milner (1963), Miyake et al. (2000).
    """
    blocks = WCST_BLOCKS["blocks"]
    all_results = []
    for block in blocks:
        prompt = _format_block_prompt(block)
        try:
            raw = llm.prompt(prompt)
        except Exception:
            raw = ""
        n_test = len(block["test_trials"])
        choices = _parse_responses(raw, n_test)
        for trial, choice in zip(block["test_trials"], choices):
            correct = (choice == trial["correct_answer"])
            error_type = None
            if not correct and trial["is_post_shift"] and trial["prev_rule"]:
                old_rule_answer = _correct_ref(trial["target"], trial["prev_rule"])
                error_type = "perseverative" if choice == old_rule_answer else "non_perseverative"
            elif not correct:
                error_type = "non_perseverative"
            all_results.append({
                "block_id": block["block_id"], "correct": correct,
                "is_post_shift": trial["is_post_shift"], "error_type": error_type,
            })
    # Metrics
    n_total = len(all_results)
    accuracy = sum(1 for r in all_results if r["correct"]) / n_total if n_total else 0
    post_shift = [r for r in all_results if r["is_post_shift"]]
    perseverative = [r for r in post_shift if not r["correct"] and r["error_type"] == "perseverative"]
    perseveration_rate = len(perseverative) / len(post_shift) if post_shift else 0.0
    block_results = {}
    for r in all_results:
        bid = r["block_id"]
        block_results.setdefault(bid, {"correct": 0, "total": 0})
        block_results[bid]["total"] += 1
        if r["correct"]: block_results[bid]["correct"] += 1
    categories = sum(1 for v in block_results.values() if v["total"] > 0 and v["correct"]/v["total"] >= 0.66)
    categories_norm = categories / len(block_results) if block_results else 0
    score = round(float(np.clip(
        0.25 * accuracy + 0.45 * (1.0 - perseveration_rate) + 0.30 * categories_norm,
        0, 1)), 4)
    _safe_log({"benchmark": "WCST_v2", "accuracy": round(accuracy, 4),
               "perseveration_rate": round(perseveration_rate, 4),
               "categories_norm": round(categories_norm, 4), "composite_score": score})
    return score

exec_func_wcst.run(llm=kbench.llm)
